# Wenu Example 1: First Chart

This notebook introduces the basic workflow for creating an astronomical chart
with Wenu.

The final chart will include:

- stars from the Hipparcos catalogue;
- western constellation lines and labels;
- IAU constellation boundaries;
- the ecliptic;
- the Galactic plane;
- celestial reference points;
- an equatorial coordinate grid.

The example keeps the main parts of the process separate:

1. define the observer;
2. construct the celestial scene;
3. choose a projection;
4. draw the chart.

This makes it easier to understand how Wenu assembles a chart and how each
component can later be customized.

## Imports

Skyfield provides the time scale and planetary ephemeris used to define the
observer. Matplotlib provides the drawing canvas.

Wenu supplies the observer, celestial sphere, coordinate grids, astronomical
data resources, and stereographic projection.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
from matplotlib.patches import Circle
from skyfield.api import Loader

from wenu.observer import Observer
from wenu.projection import StereographicProjection
from wenu.renderers import layers
from wenu.resources import (
    boundary_path,
    catalog_path,
    constellation_lines_path,
)
from wenu.sky import CelestialSphere
from wenu.sky.coordinate_grids import EclipticGrid, GalacticGrid

from wenu import Viewport
from wenu.renderers.matplotlib_viewport import apply_viewport


## Define the chart

The first step is to define the observer and the appearance of the chart.

The parameters below specify the observing location, date and time, limiting
stellar magnitude, stereographic projection, and a few visual properties of
the chart.

Later notebooks will explore how each of these parameters affects the final
result.

In [ ]:

# ---------------------------------------------------------------------
# Chart
# ---------------------------------------------------------------------

MAGNITUDE_LIMIT = 5.5

PROJECTION_RADIUS = 2.0
FLIP_EAST_WEST = True

SKY_COLOR = "slateblue"

SELECTED_CONSTELLATIONS = None

RIGHT_ASCENSIONS = range(0, 360, 30)
DECLINATIONS = (-60, -30, 0, 30, 60)

## Create the observer

An astronomical chart depends on both the observing location and the time.

Wenu stores this information in an `Observer` object. The observer also
provides the coordinate transformations needed to convert celestial positions
into altitude and azimuth.

In [ ]:
observer = Observer(
    location="La Ligua",
    time="2026-08-15 21:00",
)

In [ ]:
print("UTC time:", observer.t.utc_iso())
print("Latitude:", observer.lat_deg)
print("Longitude:", observer.lon_deg)
print("Elevation:", observer.elevation_m, "m")

## Create the celestial sphere

The `CelestialSphere` is the central object in Wenu.

It stores the astronomical objects and reference structures that will appear
in the chart. As new layers are added, they become part of the scene that
will later be projected and drawn.

In this notebook we will add:

- stars;
- constellation lines;
- constellation boundaries;
- celestial reference points;
- the ecliptic;
- the Galactic plane.

In [ ]:
sky = CelestialSphere(observer)

## Add the stars

Stars are usually the first astronomical layer to be added to the celestial
sphere.

This example uses the Hipparcos catalogue and displays all stars brighter than
magnitude 5.5.

In [ ]:
stars = sky.add_stars(
    catalog="hipparcos",
    magnitude_limit=MAGNITUDE_LIMIT,
)

The `Stars` object loads the selected catalogue, computes the apparent
positions of the stars for the observer, and prepares them for plotting.

Other catalogues can be added to Wenu in exactly the same way.

## Add the constellations

Constellation figures connect selected stars into recognizable patterns.

Wenu keeps constellation figures separate from the stellar catalogue. The
figures refer to catalogue identifiers, so the stars must be added before the
constellation layer.

This example uses the western constellation system.

In [ ]:
constellations = sky.add_constellations(
    system="western",
    selected=SELECTED_CONSTELLATIONS,
)

When `SELECTED_CONSTELLATIONS` is `None`, Wenu includes all available
constellations in the selected system.

Later notebooks will show how to draw only a chosen group of constellations.

## Add the constellation boundaries

The official IAU boundaries divide the celestial sphere into 88 constellation
regions.

These boundaries are independent of the constellation figures. A constellation
figure is a cultural drawing, while a boundary defines an official region of
the sky.

In [ ]:
boundaries = sky.add_constellation_boundaries(
    boundaries="iau",
    constellations=SELECTED_CONSTELLATIONS,
)

boundaries.sample()

## Add celestial reference points

Besides stars and constellations, astronomical charts often include reference
points that help orient the observer.

Wenu provides a collection of commonly used celestial reference points,
including the celestial poles, ecliptic poles, Galactic center, and the
cardinal points of the ecliptic.

In [ ]:
points = sky.add_points()

The visible celestial pole depends on the observer's latitude. Since this
example is for the southern hemisphere, the South Celestial Pole will be
plotted automatically.

In [ ]:
points.add_equatorial_pole(
    pole="visible",
    marker="+",
    label="SCP",
    size=120,
    color="white",
)

Next we add a few additional reference points that are frequently shown on
astronomical charts.

In [ ]:
points.add_ecliptic_pole(
    pole="south",
    marker="+",
    label="SEP",
    size=80,
    color="yellow",
)

points.add_galactic_center(
    marker="+",
    label="GC",
    size=80,
    color="lightblue",
)

points.add_ecliptic_keypoints(
    marker="+",
    size=70,
    color="cyan",
)

## Add the ecliptic

The ecliptic is the apparent annual path of the Sun across the celestial
sphere.

Wenu constructs it in ecliptic coordinates and then transforms it to the
observer's apparent sky.

In [ ]:
ecliptic_grid = EclipticGrid(
    observer=observer,
    equinox="of_date",
)

ecliptic = ecliptic_grid.ecliptic()

The `EclipticGrid` object can also be used to construct ecliptic meridians and
latitude circles. Here we use only its principal curve: ecliptic latitude
zero.

## Add the Galactic plane

The Galactic plane traces the central plane of the Milky Way.

It is defined by Galactic latitude zero and is transformed to the observer's
apparent sky in the same way as the ecliptic.

In [ ]:
galactic_grid = GalacticGrid(
    observer=observer,
)

galactic_plane = galactic_grid.galactic_plane()

The astronomical scene is now complete.

It contains stars, constellation figures, official boundaries, celestial
reference points, the ecliptic, and the Galactic plane.

The next step is to choose how this celestial sphere will be projected onto
the plane of the chart.

## Choose a projection

The celestial sphere is a three-dimensional surface, whereas a chart is a
two-dimensional drawing.

A projection defines how positions on the celestial sphere are mapped onto the
plane of the chart.

In this example we use a stereographic projection centered on the zenith. This
projection preserves angles and represents every great circle as either a
circle or a straight line, making it particularly well suited for astronomical
charts.

In [ ]:
projection = StereographicProjection(
    radius=PROJECTION_RADIUS,
    flip_ew=FLIP_EAST_WEST,
)

## Create the drawing canvas

Wenu is responsible for the astronomical calculations and rendering of the
celestial scene. In this notebook we use Matplotlib to provide the drawing
canvas on which the chart will be displayed.

The visible sky is represented by a circular horizon.

In [ ]:
fig, ax = plt.subplots(figsize=(10, 10))

fig.patch.set_alpha(0)

#ax.set_aspect("equal")
ax.axis("off")

#ax.set_xlim(
#    -1.05 * PROJECTION_RADIUS,
#     1.05 * PROJECTION_RADIUS,
#)

#ax.set_ylim(
#    -1.05 * PROJECTION_RADIUS,
#     1.05 * PROJECTION_RADIUS,
#)
R = 1.05 * PROJECTION_RADIUS
viewport = Viewport.centered(
    width=2.0 * R,
    height=2.0 * R,
)

apply_viewport(
    ax,
    viewport,
)

horizon = Circle(
    (0, 0),
    PROJECTION_RADIUS,
    facecolor=SKY_COLOR,
    edgecolor="black",
    linewidth=1.5,
)

_ = ax.add_patch(horizon)
plt.close(fig) # Let us do not draw anything yet...

## Draw the celestial sphere

Everything is now in place.

The celestial sphere contains the astronomical objects, the projection maps
them onto the plane, and Matplotlib provides the drawing surface.

Rendering the chart simply consists of asking each layer to draw itself using
the chosen projection.

In [ ]:
# Stars, constellation lines, labels, and boundaries
sky.draw(
    ax=ax,
    projection=projection,
)

In [ ]:
sky.draw_equatorial_grid(
    ax=ax,
    projection=projection,
    ra=RIGHT_ASCENSIONS,
    dec=DECLINATIONS,
    color="white",
    linewidth=0.4,
    alpha=0.35,
)

In [ ]:
ecliptic.draw(
    ax=ax,
    projection=projection,
    min_altitude=0.0,
    color="orange",
    linewidth=1.2,
)

In [ ]:
galactic_plane.draw(
    ax=ax,
    projection=projection,
    min_altitude=0.0,
    color="lightblue",
    linewidth=1.2,
)

In [ ]:
points.draw(
    ax=ax,
    projection=projection,
)

In [ ]:
ax.set_title(
    "Southern Sky\n15 August 2026, 21:00 Chile",
    fontsize=16,
)

display(fig)

## Save the chart

The completed chart can be exported as a high-resolution raster image or as a
vector graphic for publication.

In [ ]:
output_file = Path("first_chart.png")

fig.savefig(
    output_file,
    dpi=300,
    bbox_inches="tight",
    pad_inches=0.05,
)

print(f"Saved to: {output_file.resolve()}")